# Parte 3 – Modelagem, Avaliação e Explicabilidade
## Identificação de Perfis de Risco em Apostadores Online por Meio de Aprendizado de Máquina


**Integrante:** Livia Negrucci Cantowitz  RA 10389419 

**Síntese:** treina e compara algoritmos de classificação supervisionada para estimar o perfil de risco
(baixo / médio / alto) de apostadores.

**Histórico de alterações:**

| Data | Autor | Descrição |
|---|---|---|
| 20/11/2026 | Livia Negrucci Cantowitz | tudo |

## 1. Configuração e carga dos dados

In [1]:
import sys, json, warnings
from pathlib import Path
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import shap

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate, GridSearchCV
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.dummy import DummyClassifier
from sklearn.cluster import KMeans
from scipy.stats import binomtest
from sklearn.base import clone
from xgboost import XGBClassifier
from sklearn.metrics import (accuracy_score, f1_score, classification_report, confusion_matrix,
                             ConfusionMatrixDisplay, roc_auc_score, RocCurveDisplay,
                             adjusted_rand_score, silhouette_score)

sys.path.insert(0, "src")
from preparacao_dados import (carregar_apostas, agregar_por_apostador, historico_parcial,
                              calcular_risk_score, rotular_perfil, ATRIBUTOS, VARS_REGRA, PERFIS)

SEED = 42
FIG = Path("figuras"); FIG.mkdir(exist_ok=True)
Path("modelos").mkdir(exist_ok=True)
Path("resultados").mkdir(exist_ok=True)
sns.set_style("whitegrid")
ORDEM = {p: i for i, p in enumerate(PERFIS)}  # baixo=0, medio=1, alto=2

In [ ]:
apostas = carregar_apostas()
base = agregar_por_apostador(apostas)
base["risk_score"] = calcular_risk_score(base)
base["risk_profile"], cortes_completo = rotular_perfil(base["risk_score"])

# Verificação: o módulo reproduz exatamente a base entregue na Parte 2
original = pd.read_csv("apostadores_perfil_risco.csv")
print("Perfis idênticos à Parte 2:", (original["risk_profile"].values == base["risk_profile"].values).all())
print("Apostas:", len(apostas), "| Apostadores:", len(base))
print(base["risk_profile"].value_counts())
y = base["risk_profile"].map(ORDEM).values

## 2. Experimento E1 – Aprender a regra com o histórico completo


In [ ]:
def modelos_base():
    return {
        "Dummy (classe majoritária)": DummyClassifier(strategy="most_frequent"),
        "Regressão Logística": make_pipeline(StandardScaler(), LogisticRegression(max_iter=3000)),
        "Árvore de Decisão": DecisionTreeClassifier(max_depth=6, random_state=SEED),
        "k-NN (k=15)": make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=15)),
        "Random Forest": RandomForestClassifier(n_estimators=300, random_state=SEED, n_jobs=-1),
        "XGBoost": XGBClassifier(n_estimators=300, max_depth=4, learning_rate=0.05, random_state=SEED),
        "MLP (rede neural)": make_pipeline(StandardScaler(), MLPClassifier((32, 16), max_iter=1500, random_state=SEED)),
    }

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)

def comparar(X, y):
    linhas = []
    for nome, m in modelos_base().items():
        r = cross_validate(m, X, y, cv=cv, scoring=["accuracy", "f1_macro"])
        linhas.append({"Modelo": nome,
                       "Acurácia (média)": r["test_accuracy"].mean(),
                       "F1-macro (média)": r["test_f1_macro"].mean(),
                       "F1-macro (desvio)": r["test_f1_macro"].std()})
    return pd.DataFrame(linhas).sort_values("F1-macro (média)", ascending=False).round(3)

res_e1 = comparar(base[ATRIBUTOS], y)
res_e1.to_csv("resultados/resultados_E1.csv", index=False)
res_e1

## 3. Experimento E2 (principal) – Detecção precoce


In [ ]:
FRACAO = 0.5
parcial = (agregar_por_apostador(historico_parcial(apostas, FRACAO))
           .set_index("user_id").loc[base["user_id"]].reset_index())
X2 = parcial[ATRIBUTOS]

idx_tr, idx_te = train_test_split(np.arange(len(base)), test_size=0.2, stratify=y, random_state=SEED)
Xtr, Xte, ytr, yte = X2.iloc[idx_tr], X2.iloc[idx_te], y[idx_tr], y[idx_te]
print("Treino:", Xtr.shape, "| Teste:", Xte.shape)
print("Média de apostas observadas por apostador:", parcial["n_bets"].mean().round(1),
      "de", base["n_bets"].mean().round(1))

### 3.1 Linha de base: regra aplicada ao histórico parcial

In [ ]:
def regra_parcial(X_ref, X_alvo):
    # z-scores e cortes de tercil calculados apenas na referência (treino), sem vazamento
    score_ref = calcular_risk_score(X_ref)
    _, cortes = rotular_perfil(score_ref)
    rot, _ = rotular_perfil(calcular_risk_score(X_alvo, referencia=X_ref), limites=cortes)
    return rot.map(ORDEM).values

pred_regra = regra_parcial(Xtr, Xte)
print("Regra no histórico parcial (teste) – acurácia: %.3f | F1-macro: %.3f"
      % (accuracy_score(yte, pred_regra), f1_score(yte, pred_regra, average="macro")))

### 3.2 Comparação dos algoritmos (validação cruzada 5-fold no treino)

In [ ]:
res_e2 = comparar(Xtr, ytr)
res_e2.to_csv("resultados/resultados_E2_cv.csv", index=False)
res_e2

### 3.3 Ajuste de hiperparâmetros (GridSearchCV, F1-macro) dos três melhores candidatos

In [ ]:
grades = {
    "Regressão Logística": (make_pipeline(StandardScaler(), LogisticRegression(max_iter=3000)),
                            {"logisticregression__C": [0.01, 0.1, 1, 10, 100]}),
    "XGBoost": (XGBClassifier(random_state=SEED),
                {"n_estimators": [200, 400], "max_depth": [2, 3, 4], "learning_rate": [0.03, 0.1]}),
    "MLP (rede neural)": (make_pipeline(StandardScaler(), MLPClassifier(max_iter=2000, random_state=SEED)),
                          {"mlpclassifier__hidden_layer_sizes": [(16,), (32, 16)],
                           "mlpclassifier__alpha": [1e-4, 1e-2, 1]}),
}
ajustados, linhas = {}, []
for nome, (modelo, grade) in grades.items():
    gs = GridSearchCV(modelo, grade, cv=cv, scoring="f1_macro", n_jobs=-1).fit(Xtr, ytr)
    ajustados[nome] = gs.best_estimator_
    linhas.append({"Modelo": nome, "Melhores parâmetros": gs.best_params_, "F1-macro CV": round(gs.best_score_, 3)})
pd.DataFrame(linhas)

### 3.4 Avaliação final no conjunto de teste (20%, nunca usado no treino nem no ajuste)

In [ ]:
linhas = [{"Modelo": "Regra no histórico parcial (linha de base)",
           "Acurácia": accuracy_score(yte, pred_regra),
           "F1-macro": f1_score(yte, pred_regra, average="macro"),
           "Recall 'alto'": (pred_regra[yte == 2] == 2).mean(),
           "AUC alto vs. demais": np.nan}]
for nome, m in ajustados.items():
    p = m.predict(Xte); proba = m.predict_proba(Xte)
    linhas.append({"Modelo": nome, "Acurácia": accuracy_score(yte, p),
                   "F1-macro": f1_score(yte, p, average="macro"),
                   "Recall 'alto'": (p[yte == 2] == 2).mean(),
                   "AUC alto vs. demais": roc_auc_score(yte == 2, proba[:, 2])})
teste = pd.DataFrame(linhas).round(3)
teste.to_csv("resultados/resultados_E2_teste.csv", index=False)
teste

In [ ]:
melhor_nome = teste.iloc[1:].sort_values("F1-macro", ascending=False)["Modelo"].iloc[0]
melhor = ajustados[melhor_nome]
pred = melhor.predict(Xte)
print("Melhor modelo:", melhor_nome)
print(classification_report(yte, pred, target_names=PERFIS, digits=3))

fig, axes = plt.subplots(1, 2, figsize=(11, 4.2))
ConfusionMatrixDisplay(confusion_matrix(yte, pred), display_labels=PERFIS).plot(ax=axes[0], cmap="Blues", colorbar=False)
axes[0].set_title(f"Matriz de confusão – {melhor_nome}")
axes[0].grid(False); axes[0].set_xlabel("Perfil previsto"); axes[0].set_ylabel("Perfil real")
for nome, m in ajustados.items():
    RocCurveDisplay.from_predictions(yte == 2, m.predict_proba(Xte)[:, 2], name=nome, ax=axes[1])
axes[1].plot([0, 1], [0, 1], "k--", lw=0.8)
axes[1].set_title("Curva ROC – perfil 'alto' vs. demais")
axes[1].set_xlabel("Taxa de falsos positivos"); axes[1].set_ylabel("Taxa de verdadeiros positivos")
plt.tight_layout(); plt.savefig(FIG / "fig2_confusao_roc.png", dpi=200); plt.show()

### 3.4.1 A diferença para a regra é estatisticamente significativa?


In [ ]:
acerta_m, acerta_r = (melhor.predict(Xte) == yte), (pred_regra == yte)
b = int((acerta_m & ~acerta_r).sum()); c = int((~acerta_m & acerta_r).sum())
p_mcnemar = binomtest(b, b + c, 0.5).pvalue
print(f"Só o modelo acerta: {b} | só a regra acerta: {c} | McNemar exato p = {p_mcnemar:.4f}")

dif = []
for tr, va in cv.split(Xtr, ytr):
    m = clone(melhor).fit(Xtr.iloc[tr], ytr[tr])
    f_m = f1_score(ytr[va], m.predict(Xtr.iloc[va]), average="macro")
    f_r = f1_score(ytr[va], regra_parcial(Xtr.iloc[tr], Xtr.iloc[va]), average="macro")
    dif.append(f_m - f_r)
print("Ganho de F1-macro do modelo sobre a regra em cada fold:", np.round(dif, 3), "| média:", round(np.mean(dif), 3))

### 3.5 Análise de erros


In [ ]:
cm = confusion_matrix(yte, pred)
erros = cm.sum() - np.trace(cm)
extremos = cm[0, 2] + cm[2, 0]
print(f"Erros totais: {erros} | entre extremos (baixo<->alto): {extremos} ({extremos/erros:.1%} dos erros)")

analise = pd.DataFrame({"n_obs": Xte["n_bets"].values, "acerto": pred == yte,
                        "dist_corte": np.abs(base["risk_score"].values[idx_te][:, None] - cortes_completo[None, :]).min(1)})
analise["faixa_apostas"] = pd.cut(analise["n_obs"], [0, 8, 10, 12, 100], labels=["≤8", "9–10", "11–12", "≥13"])
analise["faixa_dist"] = pd.qcut(analise["dist_corte"], 4, labels=["Q1 (perto do corte)", "Q2", "Q3", "Q4 (longe)"])
display(analise.groupby("faixa_apostas", observed=True)["acerto"].agg(["mean", "size"]).round(3))
display(analise.groupby("faixa_dist", observed=True)["acerto"].agg(["mean", "size"]).round(3))

### 3.6 Quanto histórico é necessário? (curva de antecedência)

In [ ]:
curva = []
for f in [0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9]:
    Xf = (agregar_por_apostador(historico_parcial(apostas, f))
          .set_index("user_id").loc[base["user_id"]].reset_index())[ATRIBUTOS]
    m = make_pipeline(StandardScaler(), LogisticRegression(max_iter=3000, C=melhor.get_params().get("logisticregression__C", 1.0)
                                                           if melhor_nome == "Regressão Logística" else 1.0))
    m.fit(Xf.iloc[idx_tr], ytr)
    curva.append({"fração": f,
                  "Regressão Logística": f1_score(yte, m.predict(Xf.iloc[idx_te]), average="macro"),
                  "Regra no histórico parcial": f1_score(yte, regra_parcial(Xf.iloc[idx_tr], Xf.iloc[idx_te]), average="macro")})
curva = pd.DataFrame(curva).round(3)
curva.to_csv("resultados/resultados_curva_fracao.csv", index=False)

ax = curva.set_index("fração").plot(marker="o", figsize=(6.5, 4))
ax.set_xlabel("Fração inicial do histórico observada"); ax.set_ylabel("F1-macro (teste)")
ax.set_title("Detecção precoce: modelo vs. regra")
plt.tight_layout(); plt.savefig(FIG / "fig3_curva_fracao.png", dpi=200); plt.show()
curva

## 4. Explicabilidade (SHAP)


In [ ]:
fundo = Xtr.sample(300, random_state=SEED)
explicador = shap.Explainer(lambda d: melhor.predict_proba(pd.DataFrame(d, columns=ATRIBUTOS)), fundo)
sv = explicador(Xte.iloc[:500], silent=True)
sv_alto = sv[:, :, 2]

shap.plots.beeswarm(sv_alto, show=False, max_display=10)
plt.title("SHAP – contribuição para a classe 'alto risco'")
plt.tight_layout(); plt.savefig(FIG / "fig4_shap_alto.png", dpi=200, bbox_inches="tight"); plt.show()

importancia = (pd.DataFrame(np.abs(sv.values).mean(axis=0), index=ATRIBUTOS, columns=PERFIS)
               .assign(media=lambda d: d.mean(axis=1)).sort_values("media", ascending=False).round(4))
importancia.to_csv("resultados/resultados_shap_importancia.csv")
importancia

## 5. Verificação não supervisionada (k-means)


In [ ]:
Zc = StandardScaler().fit_transform(base[ATRIBUTOS])
km = KMeans(n_clusters=3, n_init=20, random_state=SEED).fit(Zc)
print("Silhueta: %.3f" % silhouette_score(Zc, km.labels_))
print("ARI (k-means vs. perfis da regra): %.3f" % adjusted_rand_score(base["risk_profile"], km.labels_))
Zr = StandardScaler().fit_transform(base[VARS_REGRA])
km_r = KMeans(n_clusters=3, n_init=20, random_state=SEED).fit(Zr)
print("ARI usando só as 5 variáveis da regra: %.3f" % adjusted_rand_score(base["risk_profile"], km_r.labels_))
display(pd.crosstab(km.labels_, base["risk_profile"]).loc[:, PERFIS])
base.assign(cluster=km.labels_).groupby("cluster")[ATRIBUTOS].mean().round(2)

## 6. Exportação do modelo para o protótipo (`app.py`)

In [ ]:
joblib.dump(melhor, "modelos/modelo_deteccao_precoce.joblib")
meta = {"modelo": melhor_nome, "fracao_historico": FRACAO, "atributos": ATRIBUTOS, "classes": PERFIS,
        "f1_macro_teste": float(teste.loc[teste.Modelo == melhor_nome, "F1-macro"].iloc[0]),
        "medianas_treino": Xtr.median().round(4).to_dict(),
        "fundo_shap": fundo.sample(100, random_state=SEED).round(4).to_dict(orient="list")}
json.dump(meta, open("modelos/metadados.json", "w"), ensure_ascii=False, indent=2)
print("Modelo salvo:", melhor_nome)

## 7. Síntese
- **E1:** com o histórico completo, a Regressão Logística reproduz a regra (F1 ≈ 0,99). Isso confirma o pipeline,
  mas não mede capacidade preditiva.
- **E2 (detecção precoce):** com 50% do histórico, a Regressão Logística (C=0,1) chega a F1-macro 0,692 no teste,
  contra 0,668 da regra aplicada ao histórico parcial. A diferença é significativa (McNemar p = 0,031)
  e o ganho é positivo nos 5 *folds* da validação cruzada. A AUC para "alto vs. demais" é 0,914. A vantagem é maior
  quando há menos histórico (+4,4 p.p. com 20%) e quase some com 90%. Só 4,2% dos erros confundem baixo com alto.
  A classe "médio" é a mais difícil (F1 0,56), e a acurácia cai para 47% nos apostadores perto dos cortes de tercil.
- **SHAP:** frequência de apostas, proporção de múltiplas, sequência de derrotas e valor apostado dominam.
  O valor máximo apostado aparece como sinal antecipado.
- **k-means:** concordância só moderada com os perfis (ARI 0,19, ou 0,33 com as 5 variáveis da regra). Os dados
  não têm três grupos naturais bem separados (silhueta 0,17), e os perfis são cortes de um contínuo de risco.
- **Limitação central:** o rótulo é uma regra definida pelo projeto, e não um diagnóstico clínico (ex.: PGSI).
  Os dados são simulados. Os resultados validam o método, mas não substituem uma validação com dados reais rotulados.